## CCS 246 Final Project: Autonomous Logistics & Resource Allocation
Course: CCS 246 - Introduction to Artificial Intelligence

> Submitted by: Asoy, Bedonia, Buenavista, Fabros, Panganiban
---

### Problem Statement

Following severe storm flooding in Metro City, emergency services face heavy traffic gridlock,
flooded roads, and unpredictable road closures. The central dispatch team must replace their static
lookup system with an intelligent dispatch agent that routes ambulances and fire engines
dynamically, repositions response hubs based on historical incident clusters, navigates dynamic road
hazards, and enforces strict city safety protocols.

---

### Project Objectives

Build four foundational AI paradigms from scratch in Python to solve operational bottlenecks in autonomous logistics, routing, and resource management:

**1. Informed Pathfinding** : Finding optimal, cost-aware travel paths across spatial grids under varying terrain/traffic conditions.<br> 
**2. Local Optimization**: Solving NP-hard spatial allocation, load scheduling, or packing problems.<br> 
**3. Adversarial Decision-Making** : Navigating against dynamic disruptions, worst-case pricing spikes, or competing agents.<br> 
**4. Knowledge-Based Logic Engine**: Evaluating complex operational policies and safety rules before executing physical or resource actions.<br> 

---

### Environment Setup

This environment initializes the Metro City spatial environment using the NetworkX graph library. It constructs the 50x50 directed grid and populates the baseline state required for all four AI modules by setting baseline road traversal costs, injecting randomized traffic congestion with weights between 3.0 and 8.0, scattering prioritized incident hotspots, mapping impassable flood boundaries, and placing the initial temporary response hubs.

In [ ]:
import math
import random
from typing import Dict, List, Optional, Tuple
import matplotlib.pyplot as plt
import networkx as nx
import numpy as np


class MetroCityEnvironment:
    """NetworkX Environment Model for Urban Emergency Dispatch & Routing.

    Integrates attributes for:
    - Module 1: Dynamic A* Search (edge weights, traversability, heuristics)
    - Module 2: Hub Location Optimization (hotspot priorities, centroid coordinates)
    - Module 3: Adversarial Navigation (hazard expansion states, dynamic blocking)
    - Module 4: Dispatch Safety Engine (depth thresholds, vehicle clearances)
    """

    def __init__(
        self,
        grid_width: int = 50,
        grid_height: int = 50,
        num_hotspots: int = 20,
        num_hubs: int = 3,
        seed: int = 42,
    ):
        np.random.seed(seed)
        random.seed(seed)

        self.width = grid_width
        self.height = grid_height
        self.num_hotspots = num_hotspots
        self.num_hubs = num_hubs

        # Instantiate NetworkX Directed Graph
        self.graph = nx.grid_2d_graph(self.width, self.height, create_using=nx.DiGraph)

        # Environment Global Attributes
        self.hubs: Dict[int, Tuple[int, int]] = {}
        self.hotspots: List[Dict] = []
        self.hazard_cells: set = set()

        # Initialize Environment Parameters
        self._initialize_node_attributes()
        self._initialize_edge_attributes()
        self.generate_hotspots()
        self.initialize_hubs()

    def _initialize_node_attributes(self) -> None:
        """Sets default node metadata across the grid."""
        for node in self.graph.nodes():
            self.graph.nodes[node]["pos"] = node
            self.graph.nodes[node]["node_type"] = "road"
            self.graph.nodes[node]["hazard_state"] = (
                "clear"  # Options: 'clear', 'flooded', 'debris'
            )
            self.graph.nodes[node]["flood_depth_m"] = 0.0  # Module 4 rule check
            self.graph.nodes[node]["hotspot_priority"] = 0  # 1 (Low) or 3 (Critical)
            self.graph.nodes[node]["is_hub"] = False

    def _initialize_edge_attributes(self) -> None:
        """Sets edge traversal costs and dynamic traversal multipliers."""
        for u, v in self.graph.edges():
            base_cost = 1.0
            congestion = 1.0
            self.graph.edges[u, v]["base_cost"] = base_cost
            self.graph.edges[u, v]["congestion_factor"] = congestion
            self.graph.edges[u, v]["weight"] = base_cost * congestion
            self.graph.edges[u, v]["is_blocked"] = False

    def generate_hotspots(self) -> None:
        """Module 2 Parameter Setup: Generates N historical incident hotspots

        with assigned emergency priority weights w_i in {1, 3}.
        """
        self.hotspots.clear()
        selected_nodes = random.sample(list(self.graph.nodes()), self.num_hotspots)

        for idx, node in enumerate(selected_nodes):
            priority = random.choice([1, 3])
            self.graph.nodes[node]["node_type"] = "hotspot"
            self.graph.nodes[node]["hotspot_priority"] = priority

            self.hotspots.append(
                {
                    "id": idx,
                    "coords": node,
                    "priority": priority,  # w_i multiplier for Module 2 optimization
                }
            )

    def initialize_hubs(self) -> None:
        """Module 2 Parameter Setup: Places K response hubs randomly initially."""
        self.hubs.clear()
        hub_nodes = random.sample(list(self.graph.nodes()), self.num_hubs)

        for k, node in enumerate(hub_nodes):
            self.hubs[k] = node
            self.graph.nodes[node]["is_hub"] = True

    def inject_traffic_congestion(
        self, congested_ratio: float = 0.15, min_cost: float = 3.0, max_cost: float = 8.0
    ) -> None:
        """Module 1 Parameter Setup: Assigns congestion costs [3.0, 8.0] to edges."""
        edges = list(self.graph.edges())
        num_congested = int(len(edges) * congested_ratio)
        congested_edges = random.sample(edges, num_congested)

        for u, v in congested_edges:
            congestion = np.random.uniform(min_cost, max_cost)
            self.graph.edges[u, v]["congestion_factor"] = congestion
            self.graph.edges[u, v]["weight"] = (
                self.graph.edges[u, v]["base_cost"] * congestion
            )

    def update_hazard_state(
        self, cell: Tuple[int, int], flood_depth: float = 0.5
    ) -> None:
        """Module 3 & 4 Parameter Update: Blocks cell and connected edges due to dynamic hazard."""
        if cell in self.graph:
            self.graph.nodes[cell]["hazard_state"] = "flooded"
            self.graph.nodes[cell]["flood_depth_m"] = flood_depth
            self.hazard_cells.add(cell)

            for u, v in list(self.graph.in_edges(cell)) + list(
                self.graph.out_edges(cell)
            ):
                self.graph.edges[u, v]["is_blocked"] = True
                self.graph.edges[u, v]["weight"] = float("inf")

    def get_manhattan_heuristic(
        self, node_a: Tuple[int, int], node_b: Tuple[int, int]
    ) -> float:
        """Module 1 Heuristic Function h_1(n): Computes Manhattan Distance."""
        return float(abs(node_a[0] - node_b[0]) + abs(node_a[1] - node_b[1]))

    def visualize_environment(self, title: str = "Metro City Graph Environment") -> None:
        """Plots the NetworkX grid with node states, hubs, hotspots, and hazards."""
        fig, ax = plt.subplots(figsize=(10, 10))
        pos = {node: node for node in self.graph.nodes()}

        road_nodes = [
            n
            for n, d in self.graph.nodes(data=True)
            if d["node_type"] == "road" and d["hazard_state"] == "clear"
        ]
        hazard_nodes = list(self.hazard_cells)
        low_pri_hotspots = [
            n for n in self.graph.nodes() if self.graph.nodes[n]["hotspot_priority"] == 1
        ]
        crit_pri_hotspots = [
            n for n in self.graph.nodes() if self.graph.nodes[n]["hotspot_priority"] == 3
        ]
        hub_nodes = [coords for coords in self.hubs.values()]

        nx.draw_networkx_nodes(
            self.graph,
            pos,
            nodelist=road_nodes,
            node_size=10,
            node_color="lightgrey",
            ax=ax,
        )
        nx.draw_networkx_nodes(
            self.graph,
            pos,
            nodelist=hazard_nodes,
            node_size=40,
            node_color="black",
            node_shape="s",
            label="Flooded/Hazard",
            ax=ax,
        )
        nx.draw_networkx_nodes(
            self.graph,
            pos,
            nodelist=low_pri_hotspots,
            node_size=50,
            node_color="blue",
            label="Hotspot (Priority 1)",
            ax=ax,
        )
        nx.draw_networkx_nodes(
            self.graph,
            pos,
            nodelist=crit_pri_hotspots,
            node_size=90,
            node_color="red",
            label="Hotspot (Priority 3)",
            ax=ax,
        )
        nx.draw_networkx_nodes(
            self.graph,
            pos,
            nodelist=hub_nodes,
            node_size=150,
            node_color="green",
            node_shape="^",
            label="Response Hub (K)",
            ax=ax,
        )

        ax.set_title(title, fontsize=14, fontweight="bold")
        ax.set_xlabel("X Grid Coordinate")
        ax.set_ylabel("Y Grid Coordinate")
        ax.grid(True, linestyle=":", alpha=0.6)
        ax.legend(loc="upper right")
        plt.tight_layout()
        plt.show()


# =============================================================================
# Execution Test
# =============================================================================
if __name__ == "__main__":
    env = MetroCityEnvironment(
        grid_width=50, grid_height=50, num_hotspots=25, num_hubs=4, seed=101
    )
    print("Environment successfully initialized without errors.")
    print(f"Nodes: {env.graph.number_of_nodes()}, Edges: {env.graph.number_of_edges()}")

In [ ]:
import matplotlib.pyplot as plt
import networkx as nx


def render_metro_city_graph(
    env: MetroCityEnvironment, title: str = "Metro City Topology & Resource Allocation"
) -> None:
    """Extracts node attributes from the MetroCityEnvironment instance and renders

    a spatial visualization of the urban grid environment.

    Visual Key:
    - Grey dots: Clear, navigable road nodes
    - Black squares: Flooded / impassable hazard cells (Module 3 & 4)
    - Blue dots: Priority 1 incident hotspots (Module 2)
    - Red dots: Priority 3 critical incident hotspots (Module 2)
    - Green triangles: K temporary response hubs (Module 2)
    """
    fig, ax = plt.subplots(figsize=(12, 12))

    # Node positions correspond directly to grid coordinates (x, y)
    pos = {node: node for node in env.graph.nodes()}

    # Categorize nodes based on attribute state
    road_nodes = [
        n
        for n, d in env.graph.nodes(data=True)
        if d["node_type"] == "road" and d["hazard_state"] == "clear"
    ]
    hazard_nodes = list(env.hazard_cells)
    low_pri_hotspots = [
        n for n in env.graph.nodes() if env.graph.nodes[n]["hotspot_priority"] == 1
    ]
    crit_pri_hotspots = [
        n for n in env.graph.nodes() if env.graph.nodes[n]["hotspot_priority"] == 3
    ]
    hub_nodes = [coords for coords in env.hubs.values()]

    # Render Node Layers
    nx.draw_networkx_nodes(
        env.graph,
        pos,
        nodelist=road_nodes,
        node_size=12,
        node_color="#d3d3d3",
        alpha=0.5,
        ax=ax,
    )

    nx.draw_networkx_nodes(
        env.graph,
        pos,
        nodelist=hazard_nodes,
        node_size=45,
        node_color="black",
        node_shape="s",
        label="Flooded / Hazard (Impassable)",
        ax=ax,
    )

    nx.draw_networkx_nodes(
        env.graph,
        pos,
        nodelist=low_pri_hotspots,
        node_size=65,
        node_color="#1f77b4",
        label="Hotspot (Priority 1: Standard)",
        ax=ax,
    )

    nx.draw_networkx_nodes(
        env.graph,
        pos,
        nodelist=crit_pri_hotspots,
        node_size=110,
        node_color="#d62728",
        label="Hotspot (Priority 3: Critical)",
        ax=ax,
    )

    nx.draw_networkx_nodes(
        env.graph,
        pos,
        nodelist=hub_nodes,
        node_size=180,
        node_color="#2ca02c",
        node_shape="^",
        edgecolors="black",
        linewidths=1.5,
        label="Response Hub ($K$)",
        ax=ax,
    )

    # Configure Axes and Grid Details
    ax.set_title(title, fontsize=15, fontweight="bold", pad=15)
    ax.set_xlabel("X Grid Coordinate", fontsize=12)
    ax.set_ylabel("Y Grid Coordinate", fontsize=12)
    ax.set_xlim(-1, env.width)
    ax.set_ylim(-1, env.height)
    ax.grid(True, linestyle="--", alpha=0.4)
    ax.legend(loc="upper right", frameon=True, facecolor="white", framealpha=0.9)

    plt.tight_layout()
    plt.show()


# =============================================================================
# Execution Block
# =============================================================================
if __name__ == "__main__":
    # 1. Initialize environment model instance
    city_env = MetroCityEnvironment(
        grid_width=50, grid_height=50, num_hotspots=25, num_hubs=4, seed=101
    )

    # 2. Inject sample hazard data (simulating dynamic flood boundary)
    for x in range(20, 26):
        for y in range(20, 26):
            city_env.update_hazard_state((x, y), flood_depth=0.8)

    # 3. Render the network graph visualization
    render_metro_city_graph(
        city_env,
        title="Metro City NetworkX Topology (Grid: 50x50, N=25 Hotspots, K=4 Hubs)",
    )

---
### Module 1 — Dynamic Route Planning & Heuristic Analysis

In [ ]:
import heapq # keeps numbers sorted automatically

class NoRouteError(Exception):
    """Raised when flooding makes the destination unreachable."""
    
    #sanity check before searching
    def plan_dynamic_route(self, origin, destination):
        
        if origin not in self.graph or destination not in self.graph:
            raise ValueError("origin and destination must be grid cells")
        if self.graph.nodes[destination]["hazard_state"] == "flooded":
            raise NoRouteError(f"destination {destination} is flooded")
        
        g_score = {origin:0.0} #cheapest minutes found so far to reach each cell
        came_from = {} # cell -> the cell we arrived from
        counter = 0 # tie breaker number
        heap = []
        h = self.get_manhattan_heuristic
        
        heapq.heappush(heap, (0.0 + h(origin, destination), counter, origin, 0.0)) # entry = (estimate f, tie-breaker, cell, minutes spent g)
        
        # the loop: pop, ignore stale entries, stop at the goal
        while heap:
            f, _, cell, g_pushed = heapq.heappop(heap)
            
            if g_pushed > g_score[cell]:
                continue # outdated entry: a better path arrived
            if cell == destination:
                break # first goal-pop is provably optimal
            
            # look at every street leaving the popped cell    
            for nxt in self.graph.successors(cell):
                edge = self.graph.edges[cell, nxt]
                if edge["is_blocked"] or edge["weight"] == float("inf"):
                    continue # flooded street: closed don't compute anything
                
                if self.graph.nodes[nxt]["hazard_state"] == "flooded":
                    continue # flooded intersection: never enter it
                
                new_g = g_score[cell] + edge["weight"] # 1.0 clear · 3.0-8.0 jam
                if new_g < g_score.get(nxt, float("inf")):
                    g_score[nxt] = new_g
                    came_from[nxt] = cell
                    counter += 1
                    heapq.heappush(heap, (new_g + h(nxt, destination), counter, nxt, new_g))
            
        # after the loop: route, breakdown, or failure
        else:
            raise NoRouteError("no route: flood wall separates origin and destination")
        
        route = [destination]
        while route[-1] != origin:
            route.append(came_from[route[-1]])
        route.reverse()
        
        breakdown = []
        for a, b in zip(route[:-1], route[1:]):
            w = self.graph.edges[a, b]["weight"]
            kind = "congested" if self.graph.edges[a, b]["congestion_factor"] > 1.0 else "clear"
            breakdown.append({"from": a, "to": b, "minutes": w, "kind": kind})
        return route, g_score[destination], breakdown
    
    MetroCityEnvironment.plan_dynamic_route = plan_dynamic_route

---
#### Module 1 Test Case: heuristic validation

The review asked for a test case that proves the A* heuristic works, with plain text output. The cell below checks h_1 against a Dijkstra baseline written in the same cell, so A* is not graded by A* itself.

Four checks, each one asserted:

1. **Admissibility**: h_1(n) stays at or under the true cheapest cost from n to the goal, tested over every reachable cell. The exact remaining costs come from a backwards search over the directed grid.
2. **Consistency**: h_1(u) <= w(u, v) + h_1(v) on every open street. This is what makes the first pop of the goal provably optimal.
3. **Optimality**: A* returns exactly the Dijkstra optimum on clear streets, under congestion (weights 3.0 to 8.0), and when a flood wall forces a detour.
4. **Effectiveness**: the number of cells expanded, measured by wrapping the graph calls, for h = 0, h_1 Manhattan, and h_2 Euclidean. Manhattan reaches the same cost after far fewer cells.

Route validity is asserted too: every hop uses an open street, never a flooded cell, and the per hop breakdown sums to the reported cost. Unreachable destinations raise NoRouteError.


In [ ]:
# =============================================================================
# Module 1 Test Case: does the Manhattan heuristic h_1(n) actually work?
# =============================================================================
# Plain text output, four checks. Each one asserts, so a broken heuristic stops
# here with an AssertionError instead of quietly passing.
#   1. Admissibility  h_1(n) never exceeds the real cheapest cost from n to goal
#   2. Consistency    h_1(u) <= w(u, v) + h_1(v) on every open street
#   3. Optimality     A* returns exactly the cost the uninformed search returns
#   4. Effectiveness  A* gets there with far fewer expansions than h = 0
# The uninformed search is written here on purpose: it is a different algorithm
# (Dijkstra), so it judges A* rather than repeating it.

EPS = 1e-9


def uninformed_uniform_cost(env, origin, destination):
    """Ground truth: the same walk with no heuristic (h = 0), i.e. Dijkstra.

    Returns the best cost. Its work is measured by count_expansions, the same
    wrapper the A* runs use.
    """
    best = {origin: 0.0}
    settled = set()
    heap = [(0.0, origin)]

    while heap:
        g, cell = heapq.heappop(heap)
        if cell in settled:
            continue
        settled.add(cell)
        if cell == destination:
            break  # first settle of the goal is optimal, so stop like A* does

        for nxt in env.graph.successors(cell):
            edge = env.graph.edges[cell, nxt]
            if edge["is_blocked"] or edge["weight"] == float("inf"):
                continue
            if env.graph.nodes[nxt]["hazard_state"] == "flooded":
                continue

            new_g = g + edge["weight"]
            if new_g < best.get(nxt, float("inf")):
                best[nxt] = new_g
                heapq.heappush(heap, (new_g, nxt))

    return best.get(destination, float("inf"))


def true_remaining_costs(env, destination):
    """Exact cheapest cost from every cell to destination.

    The same search run backwards, following predecessors instead of successors,
    because the grid is directed. Used to prove h_1 never overestimates.
    """
    best = {destination: 0.0}
    settled = set()
    heap = [(0.0, destination)]

    while heap:
        g, cell = heapq.heappop(heap)
        if cell in settled:
            continue
        settled.add(cell)

        for prev in env.graph.predecessors(cell):
            edge = env.graph.edges[prev, cell]
            if edge["is_blocked"] or edge["weight"] == float("inf"):
                continue
            if env.graph.nodes[prev]["hazard_state"] == "flooded":
                continue

            new_g = g + edge["weight"]
            if new_g < best.get(prev, float("inf")):
                best[prev] = new_g
                heapq.heappush(heap, (new_g, prev))

    return best


def count_expansions(env):
    """Counts the cells each search examines, without editing plan_dynamic_route.

    Every expansion asks the graph for the streets leaving a cell, so wrapping
    that one call measures the search exactly:
      calls = times a cell was expanded
      seen  = distinct cells expanded (the repeats are stale equal cost entries
              that pass the outdated entry check because g is equal, not larger)
    Both searches are measured through the same wrapper, so the two are comparable.
    """
    stats = {"calls": 0, "seen": set()}
    real_successors = env.graph.successors

    def counted_successors(cell):
        stats["calls"] += 1
        stats["seen"].add(cell)
        return real_successors(cell)

    env.graph.successors = counted_successors
    return stats


def reset(stats):
    stats["calls"] = 0
    stats["seen"].clear()


def report(stats):
    return len(stats["seen"]), stats["calls"] - len(stats["seen"])


def verify_route(env, origin, destination, route, cost, breakdown):
    """The returned route must be a real walk over open streets, and add up."""
    assert route[0] == origin and route[-1] == destination, "route endpoints wrong"

    for a, b in zip(route[:-1], route[1:]):
        edge = env.graph.edges[a, b]
        assert not edge["is_blocked"], f"route uses blocked street {a} -> {b}"
        assert edge["weight"] != float("inf"), f"route uses infinite street {a} -> {b}"
        assert env.graph.nodes[b]["hazard_state"] != "flooded", f"route enters flooded cell {b}"

    assert abs(sum(step["minutes"] for step in breakdown) - cost) < EPS, "breakdown sum != cost"


def heuristic_checks(env, destination, h):
    """Proves h is admissible on every reachable cell, and consistent on streets."""
    remaining = true_remaining_costs(env, destination)

    cells = 0
    tightest = float("inf")
    for cell, real_cost in remaining.items():
        overestimate = h(cell, destination) - real_cost
        assert overestimate <= EPS, f"h overestimates at {cell}: {overestimate:.3f} too high"
        tightest = min(tightest, real_cost - h(cell, destination))
        cells += 1

    streets = 0
    street_slack = float("inf")
    for u, v, data in env.graph.edges(data=True):
        if data["is_blocked"] or data["weight"] == float("inf"):
            continue
        if env.graph.nodes[v]["hazard_state"] == "flooded":
            continue
        violation = h(u, destination) - (data["weight"] + h(v, destination))
        assert violation <= EPS, f"h inconsistent on street {u} -> {v}"
        street_slack = min(street_slack, data["weight"] + h(v, destination) - h(u, destination))
        streets += 1

    return cells, tightest, streets, street_slack


def euclidean(a, b):
    """Second heuristic, used only to compare how much work each one saves."""
    return math.sqrt((a[0] - b[0]) ** 2 + (a[1] - b[1]) ** 2)


def compare_heuristics(env, origin, destination, label):
    """Runs plan_dynamic_route three times on one city: h = 0, h_1, h_2.

    Costs must all equal the uninformed optimum; unique expansions should drop.
    """
    real_h = env.get_manhattan_heuristic
    stats = count_expansions(env)
    rows = []

    for name, h in (
        ("h = 0 (Dijkstra)", lambda a, b: 0.0),
        ("h_1 Manhattan", real_h),
        ("h_2 Euclidean", euclidean),
    ):
        env.get_manhattan_heuristic = h  # plan_dynamic_route reads this attribute
        reset(stats)
        route, cost, breakdown = env.plan_dynamic_route(origin, destination)
        verify_route(env, origin, destination, route, cost, breakdown)
        rows.append((name, cost, report(stats), route, breakdown))

    env.get_manhattan_heuristic = real_h  # hand the city back untouched

    reset(stats)
    truth = uninformed_uniform_cost(env, origin, destination)
    baseline_unique, _baseline_repeats = report(stats)

    for name, cost, _counts, _route, _breakdown in rows:
        assert abs(cost - truth) < EPS, f"{name} cost {cost} != uninformed optimum {truth}"

    print(f"\n  Scenario: {label}")
    print(f"  {origin} -> {destination}, straight Manhattan {real_h(origin, destination):.2f}")
    print(f"  {'heuristic':<20} {'cost':>9} {'cells':>7} {'repeats':>8}")
    for name, cost, (unique, repeats), _route, _breakdown in rows:
        print(f"  {name:<20} {cost:>9.2f} {unique:>7} {repeats:>8}")
    print(f"  uninformed optimum {truth:.2f}, Dijkstra settled {baseline_unique} cells")
    print("  all three return the optimum: PASS")
    print(f"  Manhattan vs h = 0: {baseline_unique / rows[1][2][0]:.1f}x fewer cells expanded")
    return rows


# =============================================================================
# Execution Block
# =============================================================================
if __name__ == "__main__":
    print("=" * 74)
    print("Module 1 heuristic validation  |  h_1(n) = Manhattan distance")
    print("=" * 74)

    # --- Case 1: every street clear, h_1 is an exact lower bound -----------
    open_city = MetroCityEnvironment(
        grid_width=20, grid_height=20, num_hotspots=8, num_hubs=2, seed=101
    )
    origin, destination = (2, 3), (17, 15)
    manhattan = open_city.get_manhattan_heuristic(origin, destination)

    cells, slack, streets, street_slack = heuristic_checks(
        open_city, destination, open_city.get_manhattan_heuristic
    )
    print("\n[1] Admissibility  h_1(n) <= true remaining cost")
    print(f"    cells checked {cells}, tightest slack {slack:.2f}  -> never overestimates: PASS")
    print("[2] Consistency    h_1(u) <= w(u, v) + h_1(v)")
    print(f"    streets checked {streets}, tightest slack {street_slack:.2f}  -> monotonic: PASS")

    stats = count_expansions(open_city)
    reset(stats)
    route, cost, breakdown = open_city.plan_dynamic_route(origin, destination)
    verify_route(open_city, origin, destination, route, cost, breakdown)
    astar_unique, astar_repeats = report(stats)

    reset(stats)
    truth = uninformed_uniform_cost(open_city, origin, destination)
    dijkstra_unique, dijkstra_repeats = report(stats)

    assert abs(cost - truth) < EPS, "clear grid: A* cost != uninformed cost"
    assert abs(cost - manhattan) < EPS, "clear grid: cost should equal Manhattan exactly"
    assert len(route) == int(manhattan) + 1, "clear grid: hop count should equal Manhattan"

    print("\n[3] Clear streets (all weights 1.0)")
    print(f"    h_1 {manhattan:.2f} | A* cost {cost:.2f} | uninformed cost {truth:.2f}  -> match: PASS")
    print(f"    hops {len(route) - 1} | A* cells {astar_unique} (repeats {astar_repeats})"
          f" vs Dijkstra cells {dijkstra_unique} (repeats {dijkstra_repeats})")

    # --- Case 2: congestion 3.0-8.0, the heuristic still finds the optimum --
    jam_city = MetroCityEnvironment(
        grid_width=30, grid_height=30, num_hotspots=20, num_hubs=3, seed=101
    )
    jam_city.inject_traffic_congestion(congested_ratio=0.15, min_cost=3.0, max_cost=8.0)
    rows = compare_heuristics(
        jam_city, jam_city.hubs[0], jam_city.hotspots[0]["coords"],
        "traffic congestion on 15% of streets (weights 3.0-8.0)",
    )
    congested = sum(1 for step in rows[1][4] if step["kind"] == "congested")
    print(f"    route taken: {len(rows[1][3]) - 1} hops, {congested} of them congested")
    print("    h_1 stays admissible here because the cheapest street costs 1.0 per hop")
    print("    repeats 0 in every row means no stale heap entry re-expanded a cell")

    # --- Case 3: a flood wall forces a detour ------------------------------
    wall_city = MetroCityEnvironment(
        grid_width=20, grid_height=20, num_hotspots=5, num_hubs=1, seed=42
    )
    for y in range(20):
        if y != 0:  # leave one open corridor at the corner of the flood boundary
            wall_city.update_hazard_state((10, y), flood_depth=1.0)
    rows = compare_heuristics(
        wall_city, (3, 3), (16, 14), "flood wall at x = 10, open corridor at y = 0"
    )
    straight = wall_city.get_manhattan_heuristic((3, 3), (16, 14))
    assert rows[1][1] > straight, "detour around the wall should cost more than Manhattan"
    print(f"    straight Manhattan {straight:.2f} vs actual detour {rows[1][1]:.2f}"
          f"  -> h_1 stays under the real cost: PASS")
    print(f"    the wall costs {rows[1][1] - straight:.2f} extra minutes, h_1 never guessed above it")

    # --- Case 4: unreachable destination and bad inputs --------------------
    sealed_city = MetroCityEnvironment(
        grid_width=10, grid_height=10, num_hotspots=2, num_hubs=1, seed=7
    )
    for y in range(10):
        sealed_city.update_hazard_state((5, y), flood_depth=1.2)
    try:
        sealed_city.plan_dynamic_route((0, 0), (9, 9))
        raise AssertionError("flood wall with no corridor should be impassable")
    except NoRouteError as err:
        print(f"\n[4] Flood wall, no corridor -> NoRouteError: {err}  PASS")

    try:
        wall_city.plan_dynamic_route((3, 3), (10, 9))
        raise AssertionError("flooded destination should be refused")
    except NoRouteError as err:
        print(f"[4] Flooded destination -> NoRouteError: {err}  PASS")

    try:
        wall_city.plan_dynamic_route((99, 99), (3, 3))
        raise AssertionError("off grid origin should raise ValueError")
    except ValueError as err:
        print(f"[4] Off grid origin -> ValueError: {err}  PASS")

    print("\nAll Module 1 heuristic checks passed.")


--- 
### Module 2 — Response Hub Location Optimization

In [ ]:
# Write Code Here

---
### Module 3 — Tactical Adversarial Hazard Navigation

In [ ]:
# Write Code Here

---
### Module 4 — Automated Dispatch Safety Verification Engine

In [ ]:
# Write Code Here

---
### End-to-End Simulation

In [ ]:
# Write Code Here